# Notebook for exploring supportive components in HemOncKB

**Context:** 

If subsetting sigs table by studies such as ASCEND-4, the supportive components are not assigned a variant CUI and are assigned the variant "N/A - secondary systemic drug". This has implications for the broad category of supportive. 

**Notebook aims:**

1. Explore the components df by variant cui, checking for presence of supportive components.
2. Investigate such components, attempting to identify any implicit rules
3. Complete any further investigation required for a consistent base for sig bundle creation.

**Implied actions from findings thus far**
- Overthrow supportive categorisation approach -> REQUIRES regimen context to be accurate.
  - but not critical/asap task as does not impact treatment selection in any of our current cases

## Setup

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scoop").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the SCOOP repo root.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_ROOT = ROOT / "scoop" / "configuration"

In [2]:
import sqlalchemy as sa
import sqlalchemy.orm as so
import os

HEMONC_ENGINE_URL = os.environ["HEMONC_ENGINE_URL"]

def _require_engine_url(engine_url: str | None = None) -> str:
    resolved = engine_url or HEMONC_ENGINE_URL
    if not resolved:
        raise ValueError("Set HEMONC_ENGINE_URL or pass engine_url explicitly.")
    return resolved

engine = sa.create_engine(_require_engine_url(HEMONC_ENGINE_URL))

## Notebook body

### Components df for nsclc

In [3]:
from scoop.utils.treatment_selection import ConditionDevTools
NSCLC_BROAD_CONDITIONS = ["Lung cancer", "Non-small cell lung cancer"]

with so.Session(engine) as session:
    nsclc_helper = ConditionDevTools(conditions=NSCLC_BROAD_CONDITIONS, session=session)

In [6]:
with so.Session(engine) as session:
    variant_based_df = nsclc_helper.components.get_dataframe(session, "variant_query", True)

In [10]:
variant_based_df[variant_based_df["broad_category"]=="supportive"]

,variant_cui,id,regimen,variant_cui_1,component,component_cui,drug_inn,main_class,canmed_major_class,canmed_minor_class,study,phase,broad_category
232,144933,4263,CCI,144933.0,Mesna,328,mesna,Chemotherapy protective agent,Chemotherapy protective agent,NaN,Sculier et al. 2002,NaN,supportive
399,144713,4259,CCI,144713.0,Mesna,328,mesna,Chemotherapy protective agent,Chemotherapy protective agent,NaN,Sculier et al. 1998,NaN,supportive
400,144713,4258,CCI,144713.0,Mesna,328,mesna,Chemotherapy protective agent,Chemotherapy protective agent,NaN,Sculier et al. 1998,NaN,supportive
439,144896,18540,MIC,144896.0,Mesna,328,mesna,Chemotherapy protective agent,Chemotherapy protective agent,NaN,Crino et al. 1999,NaN,supportive
440,144896,18539,MIC,144896.0,Mesna,328,mesna,Chemotherapy protective agent,Chemotherapy protective agent,NaN,Crino et al. 1999,NaN,supportive
712,136909,14859,FULV,136909.0,Leucovorin,229,calcium folinate,Folic acid analog,Chemotherapy protective agent,NaN,Crawford et al. 1996,NaN,supportive
717,136974,18529,MIC,136974.0,Mesna,328,mesna,Chemotherapy protective agent,Chemotherapy protective agent,NaN,MIC2|Sculier et al. 2000|BTOG1,NaN,supportive
718,136974,18528,MIC,136974.0,Mesna,328,mesna,Chemotherapy protective agent,Chemotherapy protective agent,NaN,MIC2|Sculier et al. 2000|BTOG1,NaN,supportive


**Next step**
Check Leucovorin consistent with (from sigs readme)

“Required secondary systemic medications: A subset of secondary systemic medications, while not comprising the core components of a systemic anticancer treatment regimen, are nevertheless required. When this is the case, the secondary systemic medication will be labeled with the variant_cui; when there is optionality, they are instead labeled as “N/A - secondary systemic drug”. Please note that because of this requirement, a secondary systemic medication could cause the creation of a new regimen variant. For example, the regimen “Abiraterone monotherapy” has a single dosing for the primary systemic drug abiraterone but must be given with either prednisone or prednisolone.

Autologous stem cells: When used, they are always required.

Mesna: When used, it is always required.

Prednisolone: When used with abiraterone, it is required.

Prednisone: When used with abiraterone, it is required.
”

- Double checked Crawford et al. 1996 condition is nsclc  
- doi is 10.1200/JCO.1996.14.10.2774
- Leucovorin is a folic acid analog and is a primary component in FULV
- In FULV, folic acid analogs are used to INCREASE toxicity
- Therefore, the component map classification approach is flawed as supportive 
role is not a property of the component itself but how that component interacts
with other components it is administered with.